# TP4 

## Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal

## Pregunta 1

In [ ]:

# ---------------------------------------------------------------
# Parámetros
# ---------------------------------------------------------------
ARCHIVO = "mtms-arrC1A.wav"
Fc = 2000.0          # frecuencia portadora [Hz]
B = 200.0            # ancho de banda de la señal demodulada [Hz]  <-- probar distintos valores
c = 343.0            # velocidad del sonido [m/s]
T_BLOQUE = 0.1       # duración de cada bloque temporal [s]
NFFT_ESP = 256       # puntos de la FFT espacial (zero-padding)

# Posiciones de los 15 micrófonos [m], con el elemento 0 como origen
pos = np.array([0.00, 0.16, 0.24, 0.32, 0.36, 0.40, 0.44, 0.48,
                0.52, 0.56, 0.60, 0.64, 0.72, 0.80, 0.96])

# Sub-arreglos (índices de micrófonos)
subarreglos = {
    "4 cm":  [3, 4, 5, 6, 7, 8, 9, 10, 11],
    "8 cm":  [1, 2, 3, 5, 7, 9, 11, 12, 13],
    "16 cm": [0, 1, 3, 7, 11, 13, 14],
}
ELEGIDO = "8 cm"     # <-- cambiar para comparar

# ---------------------------------------------------------------
# Carga de datos
# ---------------------------------------------------------------
Fs, x = wavfile.read(ARCHIVO)          # x: (N, 15)
x = x.astype(float)
x = np.swapaxes(x, 0, 1)               # ahora (15 canales, N muestras)
N = x.shape[1]
print(f"Fs = {Fs} Hz, canales = {x.shape[0]}, duración = {N/Fs:.1f} s")

# Sub-arreglo
idx = subarreglos[ELEGIDO]
xs = np.take(x, idx, axis=0)
d = np.mean(np.diff(pos[idx]))         # espaciamiento [m]
lam = c / Fc
print(f"Sub-arreglo {ELEGIDO}: d = {d*100:.1f} cm, lambda/2 = {lam/2*100:.2f} cm")

# ---------------------------------------------------------------
# Demodulación a banda base + pasa-bajos + diezmado
# ---------------------------------------------------------------
xd = xs * np.exp(-2j * np.pi * Fc / Fs * np.arange(N))   # baja Fc a 0 Hz

sos = signal.butter(6, (B / 2) / (Fs / 2), btype="low", output="sos")
xd = signal.sosfiltfilt(sos, xd, axis=1)

M = int(Fs // B)                       # factor de diezmado (Fs_nueva >= B)
xd = xd[:, ::M]
Fs2 = Fs / M
print(f"Diezmado x{M}: nueva Fs = {Fs2:.0f} Hz, muestras = {xd.shape[1]}")

# ---------------------------------------------------------------
# Espectrograma de frecuencia espacial
# FFT a lo largo de los sensores, promediando |.|^2 sobre las muestras
# (snapshots) de cada bloque temporal.
# ---------------------------------------------------------------
K, T = xd.shape
L = int(T_BLOQUE * Fs2)                # snapshots por bloque
nb = T // L
P = np.zeros((NFFT_ESP, nb))
for b in range(nb):
    blk = xd[:, b * L:(b + 1) * L]                         # (K, L)
    X = np.fft.ifft(blk, n=NFFT_ESP, axis=0)               # FFT espacial
    P[:, b] = np.mean(np.abs(X) ** 2, axis=1)
P = np.fft.fftshift(P, axes=0)

# Eje espacial: psi = 2*pi*d*sin(theta)/lambda  ->  sin(theta) = psi*lambda/(2*pi*d)
psi = np.fft.fftshift(np.fft.fftfreq(NFFT_ESP)) * 2 * np.pi
u = psi * lam / (2 * np.pi * d)        # = sin(theta)
visible = np.abs(u) <= 1               # región física (|sin theta| <= 1)

P_dB = 10 * np.log10(P + 1e-12)
P_dB -= P_dB.max()

t = np.arange(nb) * T_BLOQUE
theta = np.degrees(np.arcsin(u[visible]))

# ---------------------------------------------------------------
# Gráfico
# ---------------------------------------------------------------
plt.figure(figsize=(10, 5))
plt.pcolormesh(t, theta, P_dB[visible, :], shading="auto", vmin=-40, vmax=0)
plt.colorbar(label="Potencia [dB]")
plt.xlabel("Tiempo [s]")
plt.ylabel("Ángulo de arribo [grados]")
plt.title(f"Espectrograma espacial - Fc={Fc:.0f} Hz, B={B:.0f} Hz, sub-arreglo {ELEGIDO}")
plt.tight_layout()
plt.show()